# Fall 2024 Data Science Track: Week 2 - Data Cleaning Exercise

## Packages, Packages, Packages!

Import *all* the things here! You need the standard stuff: `pandas` and `numpy`.

If you got more stuff you want to use, add them here too. 🙂

In [1]:
# Install pandas and numpy if the active Python environment does not have them. Using Python venv is recommended when you are doing so.
%pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np


## Introduction

With the packages out of the way, now you will be working with the following data sets:

* `food_coded.csv`: [Food choices](https://www.kaggle.com/datasets/borapajo/food-choices?select=food_coded.csv) from Kaggle
* `Ask A Manager Salary Survey 2021 (Responses) - Form Responses 1.tsv`: [Ask A Manager Salary Survey 2021 (Responses)](https://docs.google.com/spreadsheets/d/1IPS5dBSGtwYVbjsfbaMCYIWnOuRmJcbequohNxCyGVw/view?&gid=1625408792) as *Tab Separated Values (.tsv)* from Google Docs

Each one poses different challenges. But you’ll―of course―overcome them with what you learned in class! 😉

## Food Choices Data Set

### Load the Data

Load the Food choices data set into a new variable, `df_food`.

In [3]:
# Load the Food choices data set.

food_data_set_path = '../data/food_coded.csv'

df_food = pd.read_csv(food_data_set_path)

### Explore the Data

How much data did you just load?

In [4]:
len(df_food)


125

In [5]:
df_food


,GPA,Gender,breakfast,calories_chicken,calories_day,calories_scone,coffee,comfort_food,comfort_food_reasons,comfort_food_reasons_coded,...,soup,sports,thai_food,tortilla_calories,turkey_calories,type_sports,veggies_day,vitamins,waffle_calories,weight
0,2.4,2,1,430,NaN,315.0,1,none,we dont have comfort,9.0,...,1.0,1.0,1,1165.0,345,car racing,5,1,1315,187
1,3.654,1,1,610,3.0,420.0,2,"chocolate, chips, ice cream","Stress, bored, anger",1.0,...,1.0,1.0,2,725.0,690,Basketball,4,2,900,155
2,3.3,1,1,720,4.0,420.0,2,"frozen yogurt, pizza, fast food","stress, sadness",1.0,...,1.0,2.0,5,1165.0,500,none,5,1,900,I'm not answering this.
3,3.2,1,1,430,3.0,420.0,2,"Pizza, Mac and cheese, ice cream",Boredom,2.0,...,1.0,2.0,5,725.0,690,NaN,3,1,1315,"Not sure, 240"
4,3.5,1,1,720,2.0,420.0,2,"Ice cream, chocolate, chips","Stress, boredom, cravings",1.0,...,1.0,1.0,4,940.0,500,Softball,4,2,760,190
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120,3.5,1,1,610,4.0,420.0,2,"wine. mac and cheese, pizza, ice cream",boredom and sadness,NaN,...,1.0,1.0,5,940.0,500,Softball,5,1,1315,156
121,3,1,1,265,2.0,315.0,2,Pizza / Wings / Cheesecake,Loneliness / Homesick / Sadness,NaN,...,1.0,NaN,4,940.0,500,basketball,5,2,1315,180
122,3.882,1,1,720,NaN,420.0,1,"rice, potato, seaweed soup",sadness,NaN,...,1.0,2.0,5,580.0,690,none,4,2,1315,120
123,3,2,1,720,4.0,420.0,1,"Mac n Cheese, Lasagna, Pizza","happiness, they are some of my favorite foods",NaN,...,2.0,2.0,1,940.0,500,NaN,3,1,1315,135


What are the columns and their types in this data set?

In [6]:
df_food.dtypes


GPA                     str
Gender                int64
breakfast             int64
calories_chicken      int64
calories_day        float64
                     ...   
type_sports             str
veggies_day           int64
vitamins              int64
waffle_calories       int64
weight                  str
Length: 61, dtype: object

In [7]:
df_food.dtypes.reset_index().rename(columns={'index': 'column', 0: 'dtype'})


,column,dtype
0,GPA,str
1,Gender,int64
2,breakfast,int64
3,calories_chicken,int64
4,calories_day,float64
...,...,...
56,type_sports,str
57,veggies_day,int64
58,vitamins,int64
59,waffle_calories,int64


### Clean the Data

Perhaps we’d like to know more another day, but the team is really interested in just the relationship between calories (`calories_day`) and weight. …and maybe gender.

Can you remove the other columns? (Assign the result to a new variable, `df_food_col_subset`.)

In [8]:
df_food_col_subset = df_food[['calories_day', 'weight', 'Gender']].copy()
df_food_col_subset


,calories_day,weight,Gender
0,NaN,187,2
1,3.0,155,1
2,4.0,I'm not answering this.,1
3,3.0,"Not sure, 240",1
4,2.0,190,1
...,...,...,...
120,4.0,156,1
121,2.0,180,1
122,NaN,120,1
123,4.0,135,2


In [9]:
df_food_col_subset = df_food.drop(columns=[c for c in df_food.columns if c not in ['calories_day', 'weight', 'Gender']])


In [10]:
df_food_col_subset = df_food.loc[:, ['calories_day', 'weight', 'Gender']]


What about `NaN`s? How many are there?

In [11]:
df_food_col_subset.isna().sum()


calories_day    19
weight           2
Gender           0
dtype: int64

In [12]:
df_food_col_subset.isna().sum().to_frame(name='nan_count')


,nan_count
calories_day,19
weight,2
Gender,0


In [13]:
nan_counts = df_food_col_subset.isna().sum().to_frame(name='nan_count') \
    .sort_values('nan_count', ascending=False)
nan_counts


,nan_count
calories_day,19
weight,2
Gender,0


We gotta remove those `NaN`s―the entire row.

In [14]:
df_food_col_subset.dropna(inplace=True)
df_food_col_subset.shape


(104, 3)

In [15]:
# subset came from a copy so dropping rows here does not touch df_food
df_food.shape


(125, 61)

But what about the weird non-numeric values in the column obviously meant for numeric data?

Notice the data type of that column from when you got the types of all the columns?

If only we could convert the column to a numeric type and drop the rows with invalid values. 🤔

In [16]:
# weight has some junk like "144 lbs" or "Not sure, 240", coerce turns those into nan then drop them
df_food_col_subset['weight'] = pd.to_numeric(df_food_col_subset['weight'], errors='coerce')
df_food_col_subset.dropna(inplace=True)
df_food_col_subset


,calories_day,weight,Gender
1,3.0,155.0,1
4,2.0,190.0,1
5,3.0,190.0,1
6,3.0,180.0,2
7,3.0,137.0,1
...,...,...,...
118,3.0,140.0,1
119,3.0,185.0,2
120,4.0,156.0,1
121,2.0,180.0,1


Now this data seems reasonably clean for our purposes! 😁

Let’s save it somewhere to be shipped off to another teammate. 💾

In [17]:
df_food_col_subset.to_csv('../data/food_calories_weight_gender_clean.csv', index=False)


In [18]:
with open('../data/food_calories_weight_gender_clean.csv') as f:
    for i in range(5):
        print(f.readline().strip())


calories_day,weight,Gender
3.0,155.0,1
2.0,190.0,1
3.0,190.0,1
3.0,180.0,2


## Ask a Manager Salary Survey 2021 (Responses) Data Set

### Load the Data

Load the Ask A Manager Salary Survey 2021 (Responses) data set into a new variable, `df_salary`.

In [19]:
salary_data_set_path = "../data/Ask A Manager Salary Survey 2021 (Responses) - Form Responses 1.tsv"

df_salary = pd.read_csv(salary_data_set_path, sep='\t')
df_salary


,Timestamp,How old are you?,What industry do you work in?,Job title,"If your job title needs additional context, please clarify here:","What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)","How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.",Please indicate the currency,"If ""Other,"" please indicate the currency here:","If your income needs additional context, please provide it here:",What country do you work in?,"If you're in the U.S., what state do you work in?",What city do you work in?,How many years of professional work experience do you have overall?,How many years of professional work experience do you have in your field?,What is your highest level of education completed?,What is your gender?,What is your race? (Choose all that apply.)
0,4/27/2021 11:02:10,25-34,Education (Higher Education),Research and Instruction Librarian,NaN,"55,000",0.0,USD,NaN,NaN,United States,Massachusetts,Boston,5-7 years,5-7 years,Master's degree,Woman,White
1,4/27/2021 11:02:22,25-34,Computing or Tech,Change & Internal Communications Manager,NaN,"54,600",4000.0,GBP,NaN,NaN,United Kingdom,NaN,Cambridge,8 - 10 years,5-7 years,College degree,Non-binary,White
2,4/27/2021 11:02:38,25-34,"Accounting, Banking & Finance",Marketing Specialist,NaN,"34,000",NaN,USD,NaN,NaN,US,Tennessee,Chattanooga,2 - 4 years,2 - 4 years,College degree,Woman,White
3,4/27/2021 11:02:41,25-34,Nonprofits,Program Manager,NaN,"62,000",3000.0,USD,NaN,NaN,USA,Wisconsin,Milwaukee,8 - 10 years,5-7 years,College degree,Woman,White
4,4/27/2021 11:02:42,25-34,"Accounting, Banking & Finance",Accounting Manager,NaN,"60,000",7000.0,USD,NaN,NaN,US,South Carolina,Greenville,8 - 10 years,5-7 years,College degree,Woman,White
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28057,7/12/2024 22:52:01,35-44,Health care,Veterinarian,NaN,135000,NaN,USD,NaN,NaN,United States,Missouri,Wentzville,11 - 20 years,11 - 20 years,"Professional degree (MD, JD, etc.)",Woman,White
28058,7/23/2024 17:51:03,25-34,Computing or Tech,Systems Architect,NaN,109000,NaN,USD,NaN,NaN,USA,Georgia,Atlanta,5-7 years,5-7 years,College degree,Man,White
28059,7/24/2024 12:22:58,18-24,"Accounting, Banking & Finance",Risk Management Associate,NaN,1200,0.0,USD,NaN,NaN,Myanmar,Colorado,Yangon,2 - 4 years,2 - 4 years,Some college,Man,Asian or Asian American
28060,7/26/2024 11:20:45,18-24,Computing or Tech,IT,NaN,1700,10.0,USD,NaN,NaN,Burma,NaN,Yangon,2 - 4 years,1 year or less,Some college,Man,Asian or Asian American


Was that hard? 🙃

### Explore

You know the drill.

How much data did you just load?

In [20]:
len(df_salary)


28062

What are the columns and their types?

In [21]:
df_salary.dtypes


Timestamp                                                                                                                                                                                                                                   str
How old are you?                                                                                                                                                                                                                            str
What industry do you work in?                                                                                                                                                                                                               str
Job title                                                                                                                                                                                                                                   str
If your job title needs additional conte

Oh… Ugh! Give these columns easier names to work with first. 🙄

In [22]:
df_salary.columns = [
    'timestamp', 'age', 'industry', 'title', 'title_context', 'salary',
    'additional_compensation', 'currency', 'other_currency', 'salary_context',
    'country', 'state', 'city', 'total_yoe', 'field_yoe',
    'highest_education_completed', 'gender', 'race'
]
df_salary.columns


Index(['timestamp', 'age', 'industry', 'title', 'title_context', 'salary',
       'additional_compensation', 'currency', 'other_currency',
       'salary_context', 'country', 'state', 'city', 'total_yoe', 'field_yoe',
       'highest_education_completed', 'gender', 'race'],
      dtype='str')

It’s a lot, and that should not have been easy. 😏

You’re going to have a gander at the computing/tech subset first because thats *your* industry. But first, what value corresponds to that `industry`?

In [23]:
df_salary['industry'].value_counts()


industry
Computing or Tech                               4699
Education (Higher Education)                    2464
Nonprofits                                      2419
Health care                                     1896
Government and Public Administration            1889
                                                ... 
Undergrad student                                  1
Concrete Construction                              1
I'm currently a student and don't have a job       1
Student                                            1
Wine & Spirits                                     1
Name: count, Length: 1219, dtype: int64

That value among the top 5 is what you’re looking for innit? Filter out all the rows not in that industry and save it into a new variable, `df_salary_tech`. 

In [24]:
df_salary_tech = df_salary[df_salary['industry'] == 'Computing or Tech'].copy()


Do a sanity check by counting.

In [25]:
len(df_salary_tech)


4699

We are very interested in salary figures. But how many dollars 💵 is a euro 💶 or a pound 💷? That sounds like a problem for another day. 🫠

For now, let’s just look at U.S. dollars (`'USD'`).

In [26]:
df_salary_tech = df_salary_tech[df_salary_tech['currency'] == 'USD'].copy()


What we really want know is how each U.S. state pays in tech. What value in `country` represents the United States of America?

In [27]:
df_salary_tech['country'].value_counts()


country
United States               1576
USA                         1222
US                           412
U.S.                         108
United States of America      90
                            ... 
Ghana                          1
Nigeria                        1
ss                             1
Nigeria                        1
Burma                          1
Name: count, Length: 76, dtype: int64

### Clean the Data

Well, we can’t get our answers with what we currently have, so you’ll have to make some changes.

Let’s not worry about anything below the first 5 values for now. Convert the top 5 to a single canonical value―say, `'US'`, which is nice and short.

In [28]:
top_countries = df_salary_tech['country'].value_counts().head(5).index.tolist()
df_salary_tech['country'] = df_salary_tech['country'].replace(top_countries, 'US')


Have a look at the count of each unique country again now.

In [29]:
df_salary_tech['country'].value_counts()


country
US                3408
United States       68
Usa                 59
USA                 56
usa                 28
                  ... 
Ghana                1
Nigeria              1
ss                   1
Nigeria              1
Burma                1
Name: count, Length: 72, dtype: int64

Did you notice anything interesting?

In [30]:
# catch the leftover us variants like extra spaces, dots, casing, or "united state(s) of america" typos
us_pattern = r'^(u\.?s\.?a?\.?|united\s*state[s]?(\s*of\s*america)?)$'
looks_like_us = df_salary_tech['country'].str.strip().str.match(us_pattern, case=False, na=False)
df_salary_tech.loc[looks_like_us, 'country'] = 'US'


In [31]:
df_salary_tech['country'].value_counts()


country
US                      3715
Israel                     5
Canada                     4
Unite States               2
Australia                  2
India                      2
Spain                      2
Brazil                     2
United Kingdom             2
New Zealand                2
Poland                     2
France                     2
Uniyed states              1
America                    1
Puerto Rico                1
Cuba                       1
Danmark                    1
Italy                      1
International              1
United Stated              1
Remote (philippines)       1
Singapore                  1
Uruguay                    1
Mexico                     1
Canada                     1
United Stateds             1
ISA                        1
singapore                  1
Pakistan                   1
Netherlands                1
China                      1
San Francisco              1
Romania                    1
Japan                      1
United

It’s looking good so far. Let’s find out the minimum, mean, and maximum (in that order) salary by state, sorted by the mean in descending order.

In [32]:
df_salary_us_tech = df_salary_tech[df_salary_tech['country'] == 'US'].copy()

# salary is still text right now so this only gives us top/freq, not real numbers
df_salary_us_tech.groupby('state')['salary'].describe()


,count,unique,top,freq
state,,,,
Alabama,12,12,"75,000",1
"Alabama, District of Columbia",1,1,"156,000",1
"Alabama, Montana",1,1,"72,000",1
Alaska,2,2,"67,000",1
Arizona,37,34,"65,000",2
...,...,...,...,...
Vermont,8,8,"65,000",1
Virginia,117,90,"160,000",4
Washington,339,204,"160,000",14


 Well, pooh! We forgot that `salary` isn’t numeric. Something wrong must be fixed. 🤔

In [33]:
df_salary_us_tech['salary'] = df_salary_us_tech['salary'].str.replace(',', '', regex=False).astype(float)


Let’s try that again.

In [34]:
df_salary_us_tech.groupby('state')['salary'].agg(['min', 'mean', 'max']).sort_values('mean', ascending=False)


,min,mean,max
state,,,
"Michigan, Texas, Washington",340000.0,340000.000000,340000.0
"California, Oregon",200000.0,200000.000000,200000.0
"California, Colorado",176000.0,176000.000000,176000.0
"Georgia, Massachusetts",175000.0,175000.000000,175000.0
Florida,28800.0,157457.232143,2600000.0
...,...,...,...
"Massachusetts, Pennsylvania",83000.0,83000.000000,83000.0
Arkansas,55000.0,81682.300000,144000.0
"California, Maryland",81500.0,81500.000000,81500.0


That did the trick! Now let’s narrow this to data 2021 and 2022 just because (lel). *(Hint: that timestamp column may not be a temporal type right now.)*

In [35]:
df_salary_us_tech['timestamp'] = pd.to_datetime(df_salary_us_tech['timestamp'])
df_salary_us_tech_recent = df_salary_us_tech[df_salary_us_tech['timestamp'].dt.year.isin([2021, 2022, 2023])]

df_salary_us_tech_recent.groupby('state')['salary'].agg(['min', 'mean', 'max']).sort_values('mean', ascending=False)


,min,mean,max
state,,,
"Michigan, Texas, Washington",340000.0,340000.0,340000.0
"California, Oregon",200000.0,200000.0,200000.0
"California, Colorado",176000.0,176000.0,176000.0
"Georgia, Massachusetts",175000.0,175000.0,175000.0
"Alabama, District of Columbia",156000.0,156000.0,156000.0
...,...,...,...
"Massachusetts, Pennsylvania",83000.0,83000.0,83000.0
Arkansas,55000.0,81682.3,144000.0
"California, Maryland",81500.0,81500.0,81500.0


## Bonus

Clearly, we do not have enough data to produce useful figures for the level of specificity you’ve now reached. What do you notice about Delaware and West Virginia?

Let’s back out a bit and return to `df_salary` (which was the loaded data with renamed columns but *sans* filtering).

### Bonus #0

Apply the same steps as before to `df_salary`, but do not filter for any specific industry. Do perform the other data cleaning stuff, and get to a point where you can generate the minimum, mean, and maximum by state.

In [36]:
df_salary_usd = df_salary[df_salary['currency'] == 'USD'].copy()

top_countries_all = df_salary_usd['country'].value_counts().head(5).index.tolist()
df_salary_usd['country'] = df_salary_usd['country'].replace(top_countries_all, 'US')

looks_like_us_all = df_salary_usd['country'].str.strip().str.match(us_pattern, case=False, na=False)
df_salary_usd.loc[looks_like_us_all, 'country'] = 'US'

df_salary_us = df_salary_usd[df_salary_usd['country'] == 'US'].copy()
df_salary_us['salary'] = df_salary_us['salary'].str.replace(',', '', regex=False).astype(float)

df_salary_us.groupby('state')['salary'].agg(['min', 'mean', 'max']).sort_values('mean', ascending=False)


,min,mean,max
state,,,
"Michigan, Texas, Washington",340000.0,340000.000000,340000.0
"Indiana, Ohio",245000.0,245000.000000,245000.0
Alaska,27040.0,232275.078125,10000000.0
"Colorado, Nevada",190000.0,190000.000000,190000.0
"California, Montana",185000.0,185000.000000,185000.0
...,...,...,...
"Delaware, Pennsylvania",35000.0,35000.000000,35000.0
"District of Columbia, Washington",35000.0,35000.000000,35000.0
"Alabama, California",29120.0,29120.000000,29120.0


### Bonus #1

This time, format the table output nicely (*$12,345.00*) without modifying the values in the `DataFrame`. That is, `df_salary` should be identical before versus after running your code.

(*Hint: if you run into an error about `jinja2` perhaps you need to `pip install` something.*)

In [37]:
# style formats the display only, the underlying numbers in df_salary_us and df_salary are untouched
salary_stats = df_salary_us.groupby('state')['salary'].agg(['min', 'mean', 'max']).sort_values('mean', ascending=False)
salary_stats.style.format('${:,.2f}')


,min,mean,max
state,,,
"Michigan, Texas, Washington","$340,000.00","$340,000.00","$340,000.00"
"Indiana, Ohio","$245,000.00","$245,000.00","$245,000.00"
Alaska,"$27,040.00","$232,275.08","$10,000,000.00"
"Colorado, Nevada","$190,000.00","$190,000.00","$190,000.00"
"California, Montana","$185,000.00","$185,000.00","$185,000.00"
"California, Texas","$185,000.00","$185,000.00","$185,000.00"
"Georgia, Massachusetts","$175,000.00","$175,000.00","$175,000.00"
"Alabama, District of Columbia","$156,000.00","$156,000.00","$156,000.00"
"Arizona, California","$90,000.00","$152,500.00","$215,000.00"


### Bonus #2

Filter out the non-single-states (e.g., `'California, Colorado'`) in the most elegant way possible (i.e., *not* by blacklisting all the bad values).

In [38]:
# rows like "California, Colorado" are people who listed more than one state, str.contains catches all of those at once
df_salary_us_single_state = df_salary_us[~df_salary_us['state'].str.contains(',', na=False)]
df_salary_us_single_state['state'].unique()


<StringArray>
[       'Massachusetts',            'Tennessee',            'Wisconsin',
       'South Carolina',        'New Hampshire',              'Arizona',
             'Missouri',              'Florida',                    nan,
         'Pennsylvania',             'Michigan',            'Minnesota',
             'Illinois',           'California',              'Georgia',
                 'Ohio', 'District of Columbia',             'Maryland',
                'Texas',             'Virginia',       'North Carolina',
             'New York',           'New Jersey',         'Rhode Island',
             'Colorado',               'Oregon',           'Washington',
              'Indiana',                 'Iowa',             'Nebraska',
             'Oklahoma',                'Maine',          'Connecticut',
         'South Dakota',        'West Virginia',                'Idaho',
            'Louisiana',              'Montana',             'Kentucky',
         'North Dakota',             

### Bonus #3

Show the quantiles instead of just minimum, mean, and maximum―say 0%, 5%, 25%, 50%, 75%, 95%, and 100%. Outliers may be deceiving.

Sort by whatever interests you―like say the *50th* percentile.

And throw in a count by state too. It would be interesting to know how many data points contribute to the figures for each state. (*Hint: your nice formatting from Bonus #1 might not work this time around.* 😜)

In [39]:
quantile_stats = df_salary_us_single_state.groupby('state')['salary'].agg(
    count='count',
    p0=lambda s: s.quantile(0.00),
    p5=lambda s: s.quantile(0.05),
    p25=lambda s: s.quantile(0.25),
    p50=lambda s: s.quantile(0.50),
    p75=lambda s: s.quantile(0.75),
    p95=lambda s: s.quantile(0.95),
    p100=lambda s: s.quantile(1.00),
).sort_values('p50', ascending=False)
quantile_stats


,count,p0,p5,p25,p50,p75,p95,p100
state,,,,,,,,
California,2585,0.0,42000.00,72000.00,100440.0,147000.00,220000.00,875000.0
Washington,1177,72.0,40000.00,65000.00,91000.0,135000.00,200600.00,1260000.0
District of Columbia,969,40.0,48000.00,66500.00,90000.0,125000.00,189600.00,1334782.0
New York,2167,80.0,40916.00,64527.50,90000.0,127000.00,210000.00,3000000.0
Massachusetts,1514,155.0,42000.00,64000.00,86000.0,120000.00,180700.00,1650000.0
Maryland,561,0.0,40000.00,60000.00,82000.0,110000.00,165000.00,353200.0
Connecticut,236,0.0,33787.50,61750.00,81900.0,100000.00,162500.00,1900000.0
Delaware,46,35000.0,41704.00,58869.75,81322.5,105750.00,164247.50,220000.0
Virginia,780,57.0,37380.00,58750.00,80616.5,115000.00,180000.00,1300000.0
